# 🎸 BOSS IR2 - Reverse Engineering SysEx
## Notebook 1: Scoperta Dispositivi e Sniffing per Parametro

**Approccio**: Testiamo UN parametro alla volta per mappare correttamente ogni controllo.

---
## 1. Setup

In [ ]:
# !pip install mido python-rtmidi

In [ ]:
import mido
from mido import Message
import time
from datetime import datetime
import json
import os

# Crea cartella per i dati catturati
os.makedirs("captures", exist_ok=True)
print("✅ Setup completato!")

---
## 2. Trova il BOSS IR-2

In [ ]:
print("🎹 DISPOSITIVI MIDI DISPONIBILI:")
print("="*50)

print("\n📥 INPUT:")
for i, name in enumerate(mido.get_input_names()):
    boss = "  ⭐" if 'BOSS' in name.upper() or 'IR' in name.upper() else ""
    print(f"  [{i}] {name}{boss}")

print("\n📤 OUTPUT:")
for i, name in enumerate(mido.get_output_names()):
    boss = "  ⭐" if 'BOSS' in name.upper() or 'IR' in name.upper() else ""
    print(f"  [{i}] {name}{boss}")

In [ ]:
# ⚙️ CONFIGURA QUI IL TUO DISPOSITIVO
# Copia il nome esatto dalla lista sopra

IR2_INPUT = None   # es: "BOSS IR-2 MIDI 1"
IR2_OUTPUT = None  # es: "BOSS IR-2 MIDI 1"

# Auto-detect
for name in mido.get_input_names():
    if 'BOSS' in name.upper() or 'IR-2' in name.upper():
        IR2_INPUT = name
        break
        
for name in mido.get_output_names():
    if 'BOSS' in name.upper() or 'IR-2' in name.upper():
        IR2_OUTPUT = name
        break

print(f"📥 INPUT:  {IR2_INPUT}")
print(f"📤 OUTPUT: {IR2_OUTPUT}")

---
## 3. Funzioni di Utilità

In [ ]:
def hex_dump(data):
    """Formatta bytes in esadecimale"""
    return " ".join(f"{b:02X}" for b in data)

def roland_checksum(data):
    """Calcola checksum Roland"""
    return (128 - (sum(data) % 128)) % 128

def parse_sysex(data):
    """Estrae info base da SysEx Roland"""
    if len(data) < 4 or data[0] != 0x41:
        return {"hex": hex_dump(data), "type": "unknown"}
    
    return {
        "hex": hex_dump(data),
        "device_id": data[1],
        "model": hex_dump(data[2:4]),
        "command": "RQ1" if len(data) > 4 and data[4] == 0x11 else 
                   "DT1" if len(data) > 4 and data[4] == 0x12 else f"0x{data[4]:02X}" if len(data) > 4 else "?",
        "address": hex_dump(data[5:9]) if len(data) > 8 else "?",
        "payload": hex_dump(data[9:-1]) if len(data) > 10 else "?"
    }

print("✅ Funzioni caricate!")

---
## 4. 🎛️ SNIFFING PER SINGOLO PARAMETRO

**Workflow:**
1. Scrivi il nome del parametro/knob che stai per testare
2. Imposta la durata della cattura (default: 10 sec)
3. Avvia la cattura e muovi **SOLO** quel parametro
4. Attendi che finisca automaticamente

In [ ]:
# 📝 CONFIGURAZIONE CATTURA

PARAMETER_NAME = "TEST"  # ⬅️ MODIFICA QUI! Es: "VOLUME", "LOW_EQ", "CABINET"
CAPTURE_SECONDS = 10     # ⬅️ Durata cattura in secondi

print(f"🎛️ Parametro: {PARAMETER_NAME}")
print(f"⏱️ Durata: {CAPTURE_SECONDS} secondi")

In [ ]:
# 🎧 AVVIA CATTURA (si ferma automaticamente dopo CAPTURE_SECONDS)

captured = []

print(f"🎧 Cattura per: {PARAMETER_NAME}")
print("="*50)
print(f"👉 HAI {CAPTURE_SECONDS} SECONDI")
print(f"👉 Muovi SOLO il controllo '{PARAMETER_NAME}'")
print("="*50)
print()

start_time = time.time()
end_time = start_time + CAPTURE_SECONDS

with mido.open_input(IR2_INPUT) as port:
    while time.time() < end_time:
        remaining = int(end_time - time.time())
        
        # Poll con timeout breve
        msg = port.poll()
        
        if msg:
            ts = datetime.now().strftime("%H:%M:%S.%f")[:-3]
            
            if msg.type == 'sysex':
                info = parse_sysex(msg.data)
                captured.append({"time": ts, "data": list(msg.data), "info": info})
                
                print(f"[{ts}] ({remaining}s) SysEx #{len(captured)}")
                print(f"        Addr: {info.get('address', '?')} | Data: {info.get('payload', '?')}")
            else:
                captured.append({"time": ts, "type": msg.type, "raw": str(msg)})
                print(f"[{ts}] ({remaining}s) {msg.type}: {msg}")
        
        time.sleep(0.005)  # Piccola pausa per non sovraccaricare CPU

print()
print("="*50)
print(f"✅ Cattura completata! Messaggi: {len(captured)}")

In [ ]:
# 📊 ANALISI RAPIDA

sysex_only = [m for m in captured if 'data' in m]
print(f"📊 Analisi '{PARAMETER_NAME}'")
print("="*50)
print(f"Tot messaggi: {len(captured)} | SysEx: {len(sysex_only)}")
print()

if sysex_only:
    # Trova indirizzi unici
    addresses = set(m['info'].get('address', '?') for m in sysex_only)
    print(f"📍 Indirizzi usati: {addresses}")
    print()
    
    # Mostra primi e ultimi valori
    print("Primi 3 messaggi:")
    for m in sysex_only[:3]:
        print(f"  {m['info']['hex']}")
    
    if len(sysex_only) > 3:
        print("\nUltimi 3 messaggi:")
        for m in sysex_only[-3:]:
            print(f"  {m['info']['hex']}")
else:
    print("⚠️ Nessun messaggio SysEx catturato!")
    print("   Prova a muovere il parametro più lentamente.")

In [ ]:
# 💾 SALVA CATTURA

if captured:
    filename = f"captures/{PARAMETER_NAME}_{datetime.now().strftime('%H%M%S')}.json"
    with open(filename, 'w') as f:
        json.dump({
            "parameter": PARAMETER_NAME,
            "duration_seconds": CAPTURE_SECONDS,
            "timestamp": datetime.now().isoformat(),
            "messages": captured
        }, f, indent=2)
    print(f"✅ Salvato: {filename}")
else:
    print("⚠️ Nessun dato da salvare")

---
## 5. 📋 MAPPA PARAMETRI

Dopo aver testato più parametri, usa questa cella per caricare e confrontare:

In [ ]:
# Carica tutti i file catturati
import glob

param_map = {}

for filepath in glob.glob("captures/*.json"):
    with open(filepath) as f:
        data = json.load(f)
    
    param = data.get("parameter", "unknown")
    sysex_msgs = [m for m in data.get("messages", []) if 'info' in m]
    
    if sysex_msgs:
        # Prendi l'indirizzo più comune
        addrs = [m['info'].get('address', '?') for m in sysex_msgs]
        main_addr = max(set(addrs), key=addrs.count)
        param_map[param] = main_addr

print("📋 MAPPA PARAMETRI SCOPERTA:")
print("="*50)
for param, addr in sorted(param_map.items()):
    print(f"  {param:20} → {addr}")

---
## 6. Identity Request (opzionale)

In [ ]:
# Chiedi al pedale di identificarsi

print("📤 Invio Identity Request...")

with mido.open_output(IR2_OUTPUT) as out:
    with mido.open_input(IR2_INPUT) as inp:
        out.send(Message('sysex', data=[0x7E, 0x7F, 0x06, 0x01]))
        
        start = time.time()
        while time.time() - start < 2:
            msg = inp.poll()
            if msg and msg.type == 'sysex':
                print(f"\n📨 Risposta: {hex_dump(msg.data)}")
                break
            time.sleep(0.01)
        else:
            print("⚠️ Nessuna risposta (normale per alcuni dispositivi)")

---
## 📝 Checklist Parametri da Testare

Modifica `PARAMETER_NAME` e riesegui le celle 4-5 per ogni parametro:

- [ ] VOLUME
- [ ] CABINET / IR SELECT
- [ ] LOW EQ
- [ ] MID EQ  
- [ ] HIGH EQ
- [ ] MIC TYPE
- [ ] MIC DISTANCE
- [ ] PRESET 1
- [ ] PRESET 2
- [ ] PRESET 3
- [ ] ...